# Model metrics by dataset

This notebook reads every result CSV file below the results directory, normalizes the model names, and creates two images for each dataset: one for predictive metrics and another for training/prediction times. Predictive-metric images are ordered by descending macro F1; runtime images are ordered by ascending total runtime (training plus prediction). Metrics use a fixed 0–1 scale and positive times use a logarithmic scale. Missing values are displayed as a dash, while zero-shot training time is displayed as 0 s.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
from IPython.display import display

# This works whether Jupyter was started from the repository root or results/.
RESULTS_DIR = Path.cwd() / "results"
if not RESULTS_DIR.is_dir():
    RESULTS_DIR = Path.cwd()

OUTPUT_DIR = RESULTS_DIR / "metric_comparisons"
OUTPUT_DIR.mkdir(exist_ok=True)

METRICS = [
    "test_f1_macro",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_auc",
]
METRIC_LABELS = {
    "test_f1_macro": "F1 macro",
    "test_accuracy": "Accuracy",
    "test_precision": "Precision",
    "test_recall": "Recall",
    "test_auc": "AUC",
}
TIME_METRICS = ["train_time_seconds", "prediction_time_seconds"]
TIME_LABELS = {
    "train_time_seconds": "Training",
    "prediction_time_seconds": "Prediction",
}

print(f"Reading results from: {RESULTS_DIR.resolve()}")
print(f"Saving images to: {OUTPUT_DIR.resolve()}")

In [ ]:
DISPLAY_NAMES = {
    "lightgbm": "LightGBM",
    "tabicl_v2": "TabICL v2",
    "tabpfn": "TabPFN",
    "tarte": "TARTE",
    "xgboost": "XGBoost",
}


def model_label(row, source_name):
    """Build a unique, readable label for every evaluated configuration."""
    if source_name == "tabllm":
        base_model = str(row.get("model", "unknown"))
        serialization = str(row.get("serialization", "unknown"))
        return f"TabLLM: {base_model} [{serialization}]"

    if source_name == "tarte" and pd.notna(row.get("approach")):
        return f"TARTE: {row['approach']}"

    return DISPLAY_NAMES.get(source_name, source_name.replace("_", " ").title())


frames = []
result_files = sorted(RESULTS_DIR.glob("*/*_results.csv"))

for result_file in result_files:
    frame = pd.read_csv(result_file)
    if "dataset" not in frame.columns:
        print(f"Skipping {result_file}: no dataset column")
        continue

    source_name = result_file.parent.name
    frame["model_label"] = frame.apply(
        model_label, axis=1, source_name=source_name
    )
    for column in [*METRICS, *TIME_METRICS]:
        if column not in frame.columns:
            frame[column] = np.nan
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frames.append(frame[["dataset", "model_label", *METRICS, *TIME_METRICS]])

if not frames:
    raise FileNotFoundError(f"No result CSV files found under {RESULTS_DIR}")

all_results = pd.concat(frames, ignore_index=True)
all_results = all_results.sort_values(["dataset", "model_label"]).reset_index(drop=True)
print(f"Loaded {len(all_results)} result rows from {len(result_files)} files.")
display(all_results)

In [ ]:
def add_cell_grid(ax, row_count, column_count):
    ax.set_xticks(np.arange(-0.5, column_count, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, row_count, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)


def prepare_dataset_results(dataset, results):
    selected_columns = ["model_label", *METRICS, *TIME_METRICS]
    dataset_results = results.loc[results["dataset"] == dataset, selected_columns].set_index("model_label")

    # If duplicate configurations exist, retain them deterministically by averaging.
    dataset_results = dataset_results.groupby(level=0, sort=True).mean()
    # Stable sorting keeps alphabetic order when multiple models have the same F1.
    return dataset_results.sort_values(
        "test_f1_macro", ascending=False, kind="stable", na_position="last"
    )


def plot_dataset_metrics(dataset, dataset_results, output_dir=OUTPUT_DIR):
    quality_values = dataset_results[METRICS].to_numpy(dtype=float)

    figure_height = max(4.5, 0.48 * len(dataset_results) + 1.8)
    fig, quality_ax = plt.subplots(figsize=(10, figure_height), constrained_layout=True)

    quality_cmap = plt.colormaps["YlGnBu"].copy()
    quality_cmap.set_bad("white")
    quality_image = quality_ax.imshow(
        np.ma.masked_invalid(quality_values), cmap=quality_cmap,
        vmin=0, vmax=1, aspect="auto",
    )
    quality_ax.set_xticks(range(len(METRICS)), [METRIC_LABELS[m] for m in METRICS])
    quality_ax.set_yticks(range(len(dataset_results)), dataset_results.index)
    quality_ax.set_ylabel("Model")
    quality_ax.set_title(
        f"Predictive quality — {dataset.title()}\nModels ordered by macro F1",
        weight="bold", pad=12,
    )
    add_cell_grid(quality_ax, len(dataset_results), len(METRICS))

    for row_index in range(quality_values.shape[0]):
        for column_index in range(quality_values.shape[1]):
            value = quality_values[row_index, column_index]
            label = "-" if np.isnan(value) else f"{value:.3f}"
            text_color = "white" if np.isfinite(value) and value >= 0.62 else "black"
            quality_ax.text(column_index, row_index, label, ha="center", va="center", color=text_color)

    quality_colorbar = fig.colorbar(quality_image, ax=quality_ax, fraction=0.03, pad=0.02)
    quality_colorbar.set_label("Metric value")

    output_path = output_dir / f"{dataset}_model_metrics.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    return output_path


def plot_dataset_times(dataset, dataset_results, output_dir=OUTPUT_DIR):
    runtime_results = dataset_results.sort_index(kind="stable").copy()
    runtime_results["total_runtime_seconds"] = runtime_results[TIME_METRICS].sum(
        axis=1, min_count=len(TIME_METRICS)
    )
    runtime_results = runtime_results.sort_values(
        "total_runtime_seconds", ascending=True, kind="stable", na_position="last"
    )
    time_values = runtime_results[TIME_METRICS].to_numpy(dtype=float)
    positive_times = time_values[np.isfinite(time_values) & (time_values > 0)]
    figure_height = max(4.5, 0.48 * len(runtime_results) + 1.8)
    fig, time_ax = plt.subplots(figsize=(8, figure_height), constrained_layout=True)

    if positive_times.size:
        time_min, time_max = positive_times.min(), positive_times.max()
        if time_min == time_max:
            time_min, time_max = time_min / 10, time_max * 10
        time_norm = LogNorm(vmin=time_min, vmax=time_max)
        time_cmap = plt.colormaps["OrRd"].copy()
        time_cmap.set_bad("white")
        masked_times = np.ma.masked_where(~np.isfinite(time_values) | (time_values <= 0), time_values)
        time_image = time_ax.imshow(masked_times, cmap=time_cmap, norm=time_norm, aspect="auto")
        time_colorbar = fig.colorbar(time_image, ax=time_ax, fraction=0.05, pad=0.04)
        time_colorbar.set_label("Seconds (log scale)")
    else:
        time_norm = None
        time_ax.imshow(np.full_like(time_values, np.nan), cmap="OrRd", aspect="auto")

    time_ax.set_xticks(range(len(TIME_METRICS)), [TIME_LABELS[m] for m in TIME_METRICS])
    time_ax.set_yticks(range(len(runtime_results)), runtime_results.index)
    time_ax.set_ylabel("Model")
    time_ax.set_title(
        f"Runtime — {dataset.title()}\nModels ordered by total runtime (fastest to slowest)",
        weight="bold", pad=12,
    )
    add_cell_grid(time_ax, len(runtime_results), len(TIME_METRICS))

    for row_index in range(time_values.shape[0]):
        for column_index in range(time_values.shape[1]):
            value = time_values[row_index, column_index]
            if np.isnan(value):
                label, text_color = "-", "black"
            elif value <= 0:
                label, text_color = "0 s", "black"
            else:
                label = f"{value:.3g} s"
                text_color = "white" if time_norm(value) >= 0.58 else "black"
            time_ax.text(column_index, row_index, label, ha="center", va="center", color=text_color)

    output_path = output_dir / f"{dataset}_model_times.png"
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    return output_path


created_images = []
for dataset in sorted(all_results["dataset"].dropna().unique()):
    dataset_results = prepare_dataset_results(dataset, all_results)
    created_images.extend([
        plot_dataset_metrics(dataset, dataset_results),
        plot_dataset_times(dataset, dataset_results),
    ])

print("Created images:")
for image_path in created_images:
    print(f"- {image_path.resolve()}")